# TypePro Python dataset shard 00/04

Settings required: **Internet ON**, accelerator **None/CPU**. Add account
secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`. This notebook processes shard
`0` of `5` and publishes a private dataset named
`typepro-build-shard-00`.


In [ ]:
SHARD_INDEX = 0
SHARD_COUNT = 5
REPOSITORY = 'https://github.com/duyvu1105/TypePro.git'
BRANCH = 'main'
SEED = 13
TEST_PROJECTS = 100
VALIDATION_PROJECT_RATIO = 0.10
SLICE_LOG_EVERY = 50

from pathlib import Path

REPO_DIR = Path("/kaggle/working/TypePro")
WORK_DIR = Path(f"/kaggle/working/typepro_build_shard_{SHARD_INDEX:02d}")
PUBLISH_DIR = Path(f"/kaggle/working/publish_shard_{SHARD_INDEX:02d}")
print({
    "shard_index": SHARD_INDEX,
    "shard_count": SHARD_COUNT,
    "work_dir": str(WORK_DIR),
})


## Authenticate safely

Values are read from Kaggle Secrets and are never printed.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
os.environ["PYTHONUNBUFFERED"] = "1"
print("Kaggle credentials loaded for:", os.environ["KAGGLE_USERNAME"])


## Clone TypePro and install builder dependencies


In [ ]:
import shutil
import subprocess
import sys

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, REPO_DIR])
else:
    print("Using existing repository:", REPO_DIR)

PIPELINE_DIR = REPO_DIR / "codet5p_type_retrieval"
run([sys.executable, "-m", "pip", "install", "-q", "-r", PIPELINE_DIR / "requirements-build.txt"])


## Optional automatic resume

If the private shard dataset already exists, its archive is downloaded
and restored before slicing. A missing dataset simply means this is the
first run.


In [ ]:
import zipfile

dataset_id = f"{os.environ['KAGGLE_USERNAME']}/typepro-build-shard-{SHARD_INDEX:02d}"
resume_dir = Path(f"/kaggle/working/resume_shard_{SHARD_INDEX:02d}")
resume_dir.mkdir(parents=True, exist_ok=True)
probe = subprocess.run(
    ["kaggle", "datasets", "files", dataset_id],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if probe.returncode == 0 and not WORK_DIR.exists():
    run(["kaggle", "datasets", "download", "-d", dataset_id, "-p", resume_dir, "--unzip"])
    archives = list(resume_dir.glob("typepro_build_shard_*.zip"))
    if len(archives) != 1:
        raise RuntimeError(f"Expected one shard archive, found: {archives}")
    with zipfile.ZipFile(archives[0]) as bundle:
        bundle.extractall("/kaggle/working")
    print("Restored previous shard state:", WORK_DIR)
else:
    print("Starting new shard or using current working state")


## Download metadata and create the deterministic project split


In [ ]:
prepare = PIPELINE_DIR / "prepare_dataset.py"
common = [
    "--typepro-root", REPO_DIR,
    "--work-dir", WORK_DIR,
    "--split-profile", "paper_project",
    "--test-projects", TEST_PROJECTS,
    "--validation-project-ratio", VALIDATION_PROJECT_RATIO,
    "--seed", SEED,
    "--preview-samples", 1,
    "--preview-max-chars", 1200,
]
run([sys.executable, "-u", prepare, "--stage", "metadata", *common])


## Clone repositories and build interprocedural slices


In [ ]:
run([
    sys.executable, "-u", prepare,
    "--stage", "slice",
    *common,
    "--shard-count", SHARD_COUNT,
    "--shard-index", SHARD_INDEX,
    "--slice-log-every", SLICE_LOG_EVERY,
    "--build-import-kb",
    "--download-missing-imports",
    "--kb-max-files-per-package", 3000,
])


## Verify that this shard attempted every assigned project


In [ ]:
import json
sys.path.insert(0, str(PIPELINE_DIR))
from prepare_dataset import project_from_row, read_json, stable_number

projects = set()
for split in ("train", "validation", "test"):
    for row in read_json(WORK_DIR / "metadata" / f"{split}.json"):
        projects.add(project_from_row(row))
selected = {
    project for project in projects
    if stable_number(project, SEED + 4) % SHARD_COUNT == SHARD_INDEX
}
statuses = []
for path in (WORK_DIR / "project_status").glob("*.json"):
    statuses.append(json.loads(path.read_text(encoding="utf-8")))
attempted = {item.get("project") for item in statuses}
missing = sorted(selected - attempted)
summary = {
    "shard_index": SHARD_INDEX,
    "shard_count": SHARD_COUNT,
    "selected_projects": len(selected),
    "attempted_projects": len(selected & attempted),
    "successful_projects": sum(item.get("project") in selected and "error" not in item for item in statuses),
    "failed_projects": sum(item.get("project") in selected and "error" in item for item in statuses),
    "exported_slices": sum(int(item.get("exported", 0)) for item in statuses if item.get("project") in selected),
    "missing_projects": missing,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
(WORK_DIR / "shard_manifest.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
if missing:
    raise RuntimeError(f"Shard is incomplete: {len(missing)} projects missing")


## Package and publish this shard as a private Kaggle Dataset


In [ ]:
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)
archive_base = PUBLISH_DIR / f"typepro_build_shard_{SHARD_INDEX:02d}"
archive = Path(shutil.make_archive(str(archive_base), "zip", WORK_DIR.parent, WORK_DIR.name))
metadata = {
    "title": f"TypePro Python build shard {SHARD_INDEX:02d}",
    "id": dataset_id,
    "licenses": [{"name": "CC-BY-4.0"}],
}
(PUBLISH_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
exists = subprocess.run(
    ["kaggle", "datasets", "files", dataset_id],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0
if exists:
    run(["kaggle", "datasets", "version", "-p", PUBLISH_DIR, "--dir-mode", "zip", "-m", "Update completed TypePro shard"])
else:
    run(["kaggle", "datasets", "create", "-p", PUBLISH_DIR, "--dir-mode", "zip"])
print("Published:", dataset_id)
print("Archive bytes:", archive.stat().st_size)
